# Week 4 — Testing the Saved Model and Making Predictions

**Goal:** Load the trained CNN, evaluate it on test images, and run inference on unseen chest X-ray examples.

This week completes the AI workflow:

1. Load the saved model
2. Load test data
3. Evaluate model performance
4. Predict individual X-ray images
5. Interpret prediction confidence

> Educational note: This notebook is for computer vision education only. It should not be used for clinical diagnosis or medical decision-making.

## 1. Install and Import Required Packages

In [ ]:
# Install MedMNIST for PneumoniaMNIST access.
!pip install medmnist

# Libraries for visualization and data processing.
import matplotlib.pyplot as plt
import numpy as np

# TensorFlow/Keras tools for model training and loading.
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model

# Chest X-ray dataset.
from medmnist import PneumoniaMNIST

## 2. Load the Training Dataset

This section is included so the notebook can be run independently.  
If `my_model.keras` is already available from Week 3, students can skip directly to the model loading section.

In [ ]:
# Load the training split.
train_dataset = PneumoniaMNIST(split='train', download=True)

# Check dataset size.
print("Number of training images:", len(train_dataset))

## 3. Build and Compile the CNN Model

In [ ]:
# Build the same CNN architecture used in Week 3.
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(16, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),

    # Output probability:
    # close to 0 means Normal, close to 1 means Pneumonia.
    layers.Dense(1, activation='sigmoid')
])

# Compile the model before training.
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Show the model structure.
model.summary()

## 4. Prepare Training Data

This step converts images and labels into arrays.  
It is needed only if you are training the model again in this notebook.

In [ ]:
# Convert training images to normalized arrays.
x_train = np.array([
    np.array(img, dtype=np.float32) / 255.0
    for img, label in train_dataset
])

# Extract numeric labels.
y_train = np.array([
    label[0]
    for img, label in train_dataset
])

# Add the channel dimension expected by CNNs.
x_train = x_train[..., np.newaxis]

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

## 5. Train the Model

This cell trains the model and saves it.  
If you already uploaded `my_model.keras` from Week 3, you can skip this training cell to save time.

In [ ]:
# Train the CNN model.
history = model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=32
)

## 6. Visualize Training Progress

In [ ]:
# Plot training loss.
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='train loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

# Plot training accuracy.
plt.figure(figsize=(6, 4))
plt.plot(history.history['accuracy'], label='train accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training Accuracy')
plt.legend()
plt.show()

## 7. Save the Trained Model

In [ ]:
# Save the trained model for later inference.
model.save('my_model.keras')
print("Model saved!")

## 8. Load the Saved Model

Inference means using a trained model to make predictions on new data.  
Here, we load the saved model file created after training.

In [ ]:
# Load the trained model saved in Week 3 or earlier in this notebook.
model = load_model("my_model.keras")

# Confirm that the model was loaded successfully.
print("Model loaded!")

## 9. Load the Test Dataset

The **test set** contains images the model did not train on.  
Testing on unseen data gives a better sense of how well the model generalizes.

In [ ]:
# Load the test split of PneumoniaMNIST.
test_dataset = PneumoniaMNIST(split='test', download=True)

# Print the number of test images.
print("Number of test images:", len(test_dataset))

## 10. Convert Test Data to NumPy Arrays

The test data must be processed in the same way as the training data:

1. Convert images to arrays  
2. Normalize pixels to `[0, 1]`  
3. Add the channel dimension

In [ ]:
# Convert test images to normalized NumPy arrays.
x_test = np.array([
    np.array(img, dtype=np.float32) / 255.0
    for img, label in test_dataset
])

# Extract test labels.
y_test = np.array([
    label[0]
    for img, label in test_dataset
])

# Add channel dimension: (28, 28) becomes (28, 28, 1).
x_test = x_test[..., np.newaxis]

print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

## 11. Evaluate the Saved Model

Evaluation compares the model's predictions with the correct labels in the test set.

- **Test loss:** how wrong the model is on unseen data
- **Test accuracy:** how often the model predicts correctly on unseen data

In [ ]:
# Evaluate the model on the test dataset.
test_loss, test_acc = model.evaluate(x_test, y_test)

# Print test results.
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

## 12. Select and Display One Test Image

Change the `index` value to test different images.  
For example: `index = 2`, `index = 20`, or `index = 100`.

In [ ]:
# Select one test image by index.
index = 0
sample_image = x_test[index]
true_label = y_test[index]

# Display the selected X-ray image.
plt.imshow(sample_image.squeeze(), cmap='gray')
plt.title("Chest X-ray Test Image")
plt.axis('off')
plt.show()

## 13. Predict One Image

The model returns a probability between 0 and 1.

Simple interpretation rule:

- probability below `0.5` → Normal
- probability `0.5` or above → Pneumonia

In [ ]:
# Add one more dimension because the model expects a batch of images.
# Shape changes from (28, 28, 1) to (1, 28, 28, 1).
sample_input = np.expand_dims(sample_image, axis=0)

# Predict the probability of pneumonia.
prediction = model.predict(sample_input)[0][0]

# Convert probability to class label using a 0.5 threshold.
predicted_label = 1 if prediction >= 0.5 else 0

# Print prediction results.
print("Prediction score:", prediction)
print("Predicted label:", predicted_label)
print("True label:", true_label)

## 14. Show the Result as Text

Numeric labels are converted into readable class names.

In [ ]:
# Convert numeric labels to class names.
label_names = ["Normal", "Pneumonia"]

print("Predicted class:", label_names[predicted_label])
print("True class:", label_names[true_label])

## 15. Try Several Images

This final section runs inference on multiple test images and displays each prediction beside the true label.  
This helps students see examples of both correct and incorrect predictions.

In [ ]:
# Show predictions for the first 10 test images.
label_names = ["Normal", "Pneumonia"]

for i in range(10):
    image = x_test[i]
    true_label = y_test[i]

    # Add batch dimension.
    input_image = np.expand_dims(image, axis=0)

    # Predict probability and convert it to a class label.
    prediction = model.predict(input_image, verbose=0)[0][0]
    predicted_label = 1 if prediction >= 0.5 else 0

    # Display the image with predicted and true labels.
    plt.figure(figsize=(3, 3))
    plt.imshow(image.squeeze(), cmap='gray')
    plt.title(
        f"Pred: {label_names[predicted_label]} / True: {label_names[true_label]}"
    )
    plt.axis('off')
    plt.show()

## Week 4 Summary

By the end of Week 4, students learned how to:

- Load a saved Keras model
- Prepare test images
- Evaluate test loss and test accuracy
- Run inference on individual chest X-rays
- Interpret prediction scores and class labels
- Discuss how AI can support, but not replace, medical decision-making